In [0]:
# Pull the GCP credentials securely from Databricks Secrets
gcp_client_email   = dbutils.secrets.get(scope="gcp_scope", key="client_email")
gcp_project_id     = dbutils.secrets.get(scope="gcp_scope", key="project_id")
gcp_private_key    = dbutils.secrets.get(scope="gcp_scope", key="private_key")
gcp_private_key_id = dbutils.secrets.get(scope="gcp_scope", key="private_key_id")

# Set them as a configuration.
sc = spark.sparkContext
hconf = spark._jsc.hadoopConfiguration()
hconf.set("google.cloud.auth.service.account.enable", "true")
hconf.set("fs.gs.project.id", gcp_project_id)
hconf.set("fs.gs.auth.service.account.email", gcp_client_email)
hconf.set("fs.gs.auth.service.account.private.key", gcp_private_key)
hconf.set("fs.gs.auth.service.account.private.key.id", gcp_private_key_id)

In [0]:
# Paths (check these against the listing below)
BUCKET        = "gs://modelsonpaper"
ARXIV_RAW     = f"{BUCKET}/arxiv/"
ARXIV_CLEAN   = f"{BUCKET}/processed/arxiv/"

display(dbutils.fs.ls(f"{BUCKET}/"))
display(dbutils.fs.ls(ARXIV_RAW))

In [0]:
arxiv_df = spark.read.parquet(ARXIV_RAW)

arxiv_df.printSchema()
arxiv_df.show(5, truncate=50)
print("rows:", arxiv_df.count())  # expected ~3.2M

In [0]:
arxiv_rdd = arxiv_df.rdd
print("partitions:", arxiv_rdd.getNumPartitions())
arxiv_rdd.take(1)

In [0]:
print("no versions:  ", arxiv_rdd.filter(lambda r: not r.versions).count())
print("no categories:", arxiv_rdd.filter(lambda r: not r.categories).count())
print("no title:     ", arxiv_rdd.filter(lambda r: not r.title).count())

In [0]:
arxiv_rdd = arxiv_rdd.filter(lambda r: r.versions and r.categories and r.title)

In [0]:
arxiv_rows = arxiv_rdd.map(lambda r: (r.id, r.title, r.categories, r.versions[0].created))
arxiv_rows.take(5)

In [0]:
from datetime import datetime

def to_date(s):
    try:
        return datetime.strptime(s, "%a, %d %b %Y %H:%M:%S %Z").date()
    except Exception:
        return None

In [0]:
prepro_arxiv = arxiv_rows.map(lambda x: (x[0],
                                         " ".join(x[1].split()),
                                         x[2].split(" ")[0],
                                         x[2].split(" "),
                                         to_date(x[3])))
prepro_arxiv.take(3)

In [0]:
cleaned_arxiv = prepro_arxiv.map(lambda x: "\t".join([x[0], x[1], x[2], ",".join(x[3]),
                                                      str(x[4]) if x[4] else "none",
                                                      str(x[4].year) if x[4] else "none"])).cache()
cleaned_arxiv.take(3)

In [0]:
print("clean rows:   ", cleaned_arxiv.count())
print("missing dates:", prepro_arxiv.filter(lambda x: x[4] is None).count())
prepro_arxiv.map(lambda x: (x[2], 1)).reduceByKey(lambda a, b: a + b) \
            .sortBy(lambda x: x[1], ascending=False).take(10)

In [0]:
dbutils.fs.rm(ARXIV_CLEAN, True)
cleaned_arxiv.saveAsTextFile(ARXIV_CLEAN)

In [0]:
sc.textFile(ARXIV_CLEAN).count()